# 08 · Messung mit Rohdaten und Laufkontext erfassen

**Run All zeigt die lokale Vorschau.** Auswertung, Ergebnisse und Auswertungsziel sind hier nicht erforderlich.
Ein neuer Messlauf erhält die Original-CSV-Dateien als RAW_DATA und Kontext/Prüfplan als ATTACHMENT.

`config/gasi_run.json` enthält Lauf, Prüfling, Ressourcen, Kanäle, Record-/Stufenbereiche und Zeitbasis.
`config/playground_test_target.json → gasi_workflow.targets.measurement` ist das einzige Pflichtziel.
Vorbereitung: `inline` führt den Plan am Lauf; `create` registriert ihn nur bei eigenständiger Freigabe/Wiederverwendung;
`reuse` verknüpft eine vorhandene Vorbereitung als Parent.

Tagesdateien können mehrere Läufe enthalten. Im Pilot ist nur die ausdrücklich bestätigte eindeutige
Dateizuordnung `raw_file_policy.mode = exclusive_run` schreibbar. Gemeinsame Dateien bleiben bis zur
Live-Abnahme einer einmaligen Ablage gesperrt.

Eine spätere Auswertung beginnt unabhängig mit **Notebook 09**, ihren eigenen Zielen und Ergebnisdateien.
Details: [Forschungsupload](docs/GaSi_Forschungsupload.md).

In [ ]:
from pathlib import Path
import importlib
import gasi_run
from IPython.display import FileLink, display

importlib.invalidate_caches()
gasi_run = importlib.reload(gasi_run)
run_path = Path("config/gasi_run.json")
if not run_path.exists():
    with run_path.open("xb") as target:
        target.write(Path("config/gasi_run.example.json").read_bytes())
    print("Leere lokale Laufvorlage angelegt: config/gasi_run.json")
preview_sha256 = None
run = gasi_run.load_measurement(run_path)
issues = gasi_run.validate_run(run)
target_config, target_message = gasi_run.ob.load_target_config()
print(target_message)
for activity in gasi_run.activities(run):
    try:
        path = gasi_run.ob.target_path((target_config or {}).get("gasi_workflow", {}).get("targets", {}).get(activity, {}))
        print(activity + ":", path)
    except gasi_run.ob.CheckFailed:
        print(activity + ": NICHT KONFIGURIERT")
try:
    gasi_run._targets(target_config, gasi_run.activities(run))
except gasi_run.ob.CheckFailed as exc:
    issues.append("Lokale Zielprüfung: " + str(exc))
policy = (run or {}).get("raw_file_policy", {})
for item in policy.get("shared_sources", []):
    print("Gemeinsam verwendete Datei:", item.get("file"), "Läufe:", item.get("run_ids"))
if issues:
    print(f"Upload gesperrt: {len(issues)} Angaben fehlen oder sind zu prüfen.")
    for issue in issues:
        print("-", issue)
else:
    bundle = gasi_run.prepare_bundle(run)
    preview_sha256 = bundle["sha256"]
    print("Lokales Uploadpaket geprüft; noch keine Instanzbestätigung.")
    print("Schritte:", ", ".join(gasi_run.activities(run)))
    print("Prüfling und Ressourcen werden als vorhandene Objekte wiederverwendet.")
    print("Alle Originaldateien bleiben unverändert.")
    display(FileLink((bundle["path"] / "manifest.json").relative_to(Path.cwd()).as_posix(), result_html_prefix="Lokales Manifest: "))

In [ ]:
# Bewusst aktivieren und nur diese Zelle ausführen, wenn Lauf und Ziele geprüft sind.
UPLOAD_ENABLED = False
CONFIRMATION = ""  # UPLOAD CONFIGURED GASI RESEARCH WORKFLOW TO PLAYGROUND

result, message = gasi_run.upload_workflow(enabled=UPLOAD_ENABLED, confirmation=CONFIRMATION, expected_manifest_sha=globals().get("preview_sha256"))
print(message)
if result:
    for activity, entry in result["objects"].items():
        print(activity, entry["identifier"], entry["collection"], entry.get("url", ""))
    for entry in result["datasets"]:
        print(entry["activity"], entry["identifier"], "SHA-256:", entry["sha256"], entry.get("url", ""))